# Étude Approfondie : Infrastructures de Recharge (IRVE)

## 1. Introduction et Objectifs

Ce notebook est dédié à l'analyse des données IRVE (Infrastructures de Recharge pour Véhicules Électriques). Contrairement aux données INSEE, ces données sont "sales" et nécessitent un traitement important avant de pouvoir être agrégées à l'échelle communale.

L'objectif final de ce notebook est de construire un jeu de données agrégé par code géographique à partir de la base nationale des bornes de recharge électrique (IRVE), afin d'expliquer ou prédire le taux de véhicules électriques local.

## 2. Initialisation des données

Ici, chaque ligne correspond à une borne / point de recharge déclaré. L'objectif final étant une modélisation territoriale, chaque borne doit être rattachée à une zone géographique.

Pour mener cette étude, nous devons d'abord garantir l'intégrité géographique de la base IRVE. Cette étape technique permet de compléter les codes INSEE manquants via un géocodage spatial (coordonnées GPS) afin d'assurer une base de données exhaustive pour l'analyse.

In [ ]:
import pandas as pd
from src.utils import (
    creer_gdf_irve,
    joindre_communes,
    ajouter_codes_geo
)
from src.loading import load_irve_data, charger_communes
from src.cleaning import (
    nettoyer_code_insee
)

# 1. Chargement
url_irve ="https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435"
df_irve = load_irve_data(url_irve)

# 2. Préparation technique (Codes Géo)
df_irve["code_insee_commune"] = df_irve["code_insee_commune"].apply(nettoyer_code_insee)
gdf_irve = creer_gdf_irve(df_irve, "consolidated_longitude", "consolidated_latitude")
gdf_communes = charger_communes()
gdf_result = joindre_communes(gdf_irve, gdf_communes)
df_irve = ajouter_codes_geo(df_irve, gdf_result)

print(f"Base chargée et géocodée : {len(df_irve)} points de charge prêts pour l'analyse.")
df_irve.sample(5)

## 3. Sélection initiale des variables

Les variables ont été retenues selon leur potentiel explicatif :
- attractivité du réseau
- accessibilité
- performance technique
- structure concurrentielle

In [ ]:
list(df_irve.columns)

Après un premier tri des variables disponibles, nous retenons celles présentant le plus d’intérêt pour la suite de l’analyse :

In [ ]:
var_interet = [
    "code_geo_total",
    "nom_operateur",
    "implantation_station",
    "nbre_pdc",
    "puissance_nominale",
    "prise_type_ef",
    "prise_type_2",
    "prise_type_combo_ccs",
    "prise_type_chademo",
    "prise_type_autre",
    "cable_t2_attache",
    "gratuit",
    "paiement_acte",
    "paiement_cb",
    "paiement_autre",
    "tarification",
    "condition_acces",
    "reservation",
    "horaires",
    "created_at"
]

df_filtre = df_irve[var_interet].copy()

## 4. Vérification des types

Les types de données fournis par la source ne sont pas respectés dans les données brutes. Plusieurs variables booléennes et temporelles sont encodées en object, nécessitant une étape de conversion avant analyse.

In [ ]:
df_filtre.info()

Nous allons désormais corriger les types de variables afin de les aligner sur ceux indiqués dans la documentation officielle de data.gouv.fr.

#### Booléens

In [ ]:
cols_bool = [
    'prise_type_ef',
    'prise_type_2',
    'prise_type_combo_ccs',
    'prise_type_chademo',
    'prise_type_autre',
    'cable_t2_attache',
    'gratuit',
    'paiement_acte',
    'paiement_cb',
    'paiement_autre',
    'reservation'
]

# Récupérer toutes les valeurs uniques
valeurs_uniques = set()
for col in cols_bool:
    valeurs_uniques.update(df_filtre[col].unique())
print(valeurs_uniques, "\n")

mapping = {
    'true': True,
    'false': False,
    '1': True,
    '0': False
}

for col in cols_bool:
    df_filtre[col] = (
        df_filtre[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
        .astype("boolean")
    )

for col in cols_bool:
    print(f"{col} :", df_filtre[col].unique())

Après avoir identifié des incohérences de formatage dans les variables booléennes, un mapping est appliqué afin d’harmoniser leurs valeurs. Les différentes écritures observées sont ainsi converties vers un format booléen standardisé ne prenant désormais que trois modalités possibles : `True`, `False` ou `NaN`.

#### Date

In [ ]:
df_filtre['created_at'] = pd.to_datetime(df_filtre['created_at'])

#### String / catégories

In [ ]:
cols_str = [
    'nom_operateur',
    'implantation_station',
    'tarification',
    'condition_acces',
    'horaires'
]

for col in cols_str:
    df_filtre[col] = (
        df_filtre[col]
        .astype("string")
    )

df_filtre.describe(include='string[python]')

Les variables `condition_acces` et `implantation_station` présentent un faible nombre de modalités distinctes. Il peut donc être pertinent de les convertir au type `category` afin d’optimiser leur stockage et leur traitement.

Avant cette transformation, nous effectuons une analyse rapide de leur distribution afin de confirmer cette hypothèse.

In [ ]:
df_irve['condition_acces'].unique()

In [ ]:
df_irve['implantation_station'].unique()

Les variables `condition_acces` et `implantation_station` présentent plusieurs anomalies de formatage, principalement liées à des problèmes d’encodage des caractères spéciaux.

Avant toute conversion en type catégoriel, une étape de nettoyage est donc réalisée afin de :

- corriger les libellés mal encodés ;
- harmoniser les écritures multiples d’une même modalité ;
- simplifier certaines catégories pour faciliter l’analyse statistique.

---

**Nettoyage de `condition_acces`**

Plusieurs variantes incorrectes du libellé **Accès libre** ont été identifiées. Elles sont regroupées sous une seule modalité standardisée :

- `Accčs libre`
- `Accs libre`
- `Acc¸s libre`
- `AccĂ¨s libre`

sont remplacées par :

- `Accès libre`

---

**Nettoyage et regroupement de `implantation_station`**

Des erreurs d’encodage ont également été corrigées dans les libellés d’implantation des bornes.

Ensuite, les modalités ont été regroupées en quatre grandes catégories interprétables.

**Récapitulatif des catégories d’implantation des bornes IRVE**

Ces catégories décrivent le lieu d’installation ou le type d’usage des bornes de recharge.

| Catégorie | Définition | Exemples | Interprétation |
|---|---|---|---|
| **Privé** | Borne située sur un espace privé, souvent réservé à certains usagers | entreprise, hôtel, supermarché, copropriété | présence d’activités économiques ou services privés |
| **Public** | Borne installée sur un parking accessible à tous | parking municipal, gare, centre-ville | volonté locale de développer la recharge publique |
| **Voirie** | Borne installée directement sur la voie publique | rue, trottoir, stationnement urbain | utile dans les zones denses sans stationnement privé |
| **Rapide** | Station dédiée à la recharge haute puissance | hub de recharge, autoroute, station rapide | usage de transit / longs trajets |

In [ ]:
df_filtre["condition_acces"] = df_filtre["condition_acces"].replace({
    "Accčs libre": "Accès libre",
    "Accs libre": "Accès libre",
    "Acc¸s libre": "Accès libre",
    "AccĂ¨s libre": "Accès libre"
})


replacements = {
    "Parking priv rserv  la clientle": "Parking privé réservé à la clientèle",
    "Parking priv  usage public": "Parking privé à usage public"
}

df_filtre["implantation_station"] = (
    df_filtre["implantation_station"]
    .replace(replacements)
    .str.strip()
)

mapping = {
    "Parking privé réservé à la clientèle": "prive",
    "Parking privé à usage public": "prive",
    "Parking public": "public",
    "Voirie": "voirie",
    "Station dédiée à la recharge rapide": "rapide"
}

df_filtre["implantation_station_clean"] = df_filtre["implantation_station"].replace(mapping)

In [ ]:
df_filtre['implantation_station_clean'] = df_filtre['implantation_station_clean'].astype('category')
df_filtre['condition_acces'] = df_filtre['condition_acces'].astype('category')

#### Descriptif rapide de nos variables

In [ ]:
df_filtre.describe(include='all')

## 5. Analyse des valeurs manquantes

Pour la suite de l’analyse, nous commençons par supprimer la variable `implantation_station`, devenue inutile depuis la création de sa version nettoyée et regroupée.

In [ ]:
df_filtre = df_filtre.drop(columns=["implantation_station"])

In [ ]:
na = (
    df_filtre.isna()
    .sum()
    .sort_values(ascending=False)
)

na_pct = (na / len(df_filtre) * 100).round(2)

pd.DataFrame({
    "nb_manquants": na,
    "%": na_pct
}).head(10)

Les variables comportants beaucoup de valeurs manquantes pourraient être problématiques. Analysons les :

#### tarification

Au-delà du nombre important de valeurs manquantes, cette variable contient des informations sous forme de texte libre. La grande diversité des modalités rend son exploitation difficile dans un cadre d’analyse ou de modélisation.

Par conséquent, nous faisons le choix de l’écarter de l’étude.

In [ ]:
print(df_filtre["tarification"].value_counts(dropna=False).head(15))
print()
print("Nb modalités uniques :", df_filtre["tarification"].nunique())
df_filtre = df_filtre.drop(columns=["tarification"])

#### cable_t2_attache

Cette variable présente non seulement une proportion importante de valeurs manquantes (près de la moitié des observations), mais également un fort déséquilibre dans ses modalités.

Dans ce contexte, et afin d’éviter d’introduire du bruit dans l’analyse, nous faisons le choix de l’écarter de l’étude.

In [ ]:
df_filtre["cable_t2_attache"].value_counts(normalize=True) * 100
df_filtre = df_filtre.drop(columns=["cable_t2_attache"])

________________ début __________________

#### paiement_autre

In [ ]:
df_filtre["paiement_autre"].value_counts(normalize=True) * 100

________________ fin __________________

#### gratuit
Au-delà du nombre de valeurs manquantes, cette variable présente un fort déséquilibre entre ses modalités : la quasi-totalité des bornes étant payantes, l’information relative à la gratuité apporte peu de variance et donc peu de valeur explicative.

Dans ce contexte, et afin d’éviter d’introduire du bruit dans l’analyse, nous faisons le choix de l’écarter de l’étude.

In [ ]:
df_filtre["gratuit"].value_counts(normalize=True) * 100
df_filtre = df_filtre.drop(columns=["gratuit"])

________________ début __________________

#### paiement_cb

In [ ]:
df_filtre["paiement_cb"].value_counts(normalize=True) * 100

#### nom_operateur

La part de valeurs manquantes reste faible, cela ne nuira pas à notre analyse, nous décidons de garder cette variable pour le moment.

________________ fin __________________

## 6. Choix des Variables

#### Variables écartées

Commençons par réaliser quelques analyses exploratoires rapides afin d’identifier et d’écarter d’éventuelles variables non pertinentes pour la suite de l’étude.

In [ ]:
(df_filtre.select_dtypes(include="bool")
 .drop(columns=["paiement_autre", "paiement_cb"])
 .apply(lambda x: x.value_counts(normalize=True) * 100))

In [ ]:
df_filtre["condition_acces"].value_counts(normalize=True) * 100

________________ début __________________

`paiement_acte`, `reservation`, `condition_acces` est très déséquilibrée, faible pouvoir discriminant. Elle n'apporte pas beaucoup d'information donc on décide de la supprimer.
86% des observations correspondent à "Accès libre".

In [ ]:
import matplotlib.pyplot as plt
df_filtre["annee"] = df_filtre["created_at"].dt.year
df_filtre["annee"].value_counts().sort_index().plot()
plt.show()

`created_at` est davantage administrative que structurelle.

In [ ]:
# horaires
print(df_filtre["horaires"].describe())
print(df_filtre["horaires"].value_counts(dropna=False).head(15))
print("Nb modalités uniques :", df_filtre["horaires"].nunique())

Les valeurs sont trop diverses ce qui rend la variables difficile à traiter.

| Variable         | Raison                      |
| ---------------- | --------------------------- |
| cable_t2_attache | ~48% manquants              |
| horaires         | texte libre très hétérogène |
| tarification     | texte libre très hétérogène           |
| prise_type_autre | peu informative             |
| paiement_acte    | quasi constante             |
| condition_acces  | très déséquilibrée          |
| reservation      | faible variabilité          |
| created_at       | date administrative         |
| nom_amenageur| trop de modalités   |
| nom_enseigne| trop de modalités   |


## Variables retenues pour la modélisation

In [ ]:
vars_finales = ['nom_operateur',
               'implantation_station',
               'nbre_pdc',
               'puissance_nominale',
               'prise_type_ef',
               'prise_type_2',
               'prise_type_combo_ccs',
               'prise_type_chademo',
               'gratuit',
               'paiement_cb',
               'paiement_autre']

## Agrégation territoriale

Une ligne finale = un code géographique.

| Variable finale   | Construction      |
| ----------------- | ----------------- |
| total_pdc         | somme             |
| puissance_moyenne | moyenne           |
| puissance_max     | max               |
| nb_operateurs     | nunique           |
| top_operateur     | mode              |
| pct_type_2        | moyenne booléenne |
| pct_gratuit       | moyenne booléenne |
| part_voirie       | dummies + moyenne |


Le jeu de données final est désormais prêt à être fusionné avec les données socio-économiques locales afin de modéliser le taux de véhicules électriques.

# Brouillon

### puissance_nominale
faire des analyse uni pour bien comprendre les puissances des bornes et  créez une variable "Nombre de bornes ultra-rapides" (> 50 kW)

In [ ]:
# ------------------------------------------------------------
# 6. created_at
# ------------------------------------------------------------

print("===== created_at =====")
print(df_filtre["created_at"].describe())

# année création
df_filtre["annee_creation"] = df_filtre["created_at"].dt.year

print(df_filtre["annee_creation"].value_counts().sort_index())

df_filtre["annee_creation"].value_counts().sort_index().plot(kind="line", marker="o")
plt.title("Nombre de bornes par année de création")
plt.xlabel("Année")
plt.ylabel("Nombre")
plt.show()


# ancienneté en années
today = pd.Timestamp.today(tz="UTC")
df_filtre["anciennete"] = (today - df_filtre["created_at"]).dt.days / 365

print(df_filtre["anciennete"].describe())

df_filtre["anciennete"].hist(bins=30)
plt.title("Distribution ancienneté des bornes")
plt.xlabel("Ancienneté (années)")
plt.show()

1. Variables d'identification et d'acteurs
Ces variables permettent de mesurer la diversité et le type d'acteurs présents.

Représentent les entités responsables de l'installation et de l'exploitation.
Usage final : Calculez le nombre d'opérateurs différents par commune. Plus il y a d'acteurs, plus le réseau est compétitif et attractif.
['nom_amenageur',
 'nom_operateur',
 'nom_enseigne']


2. Variables de localisation et d'infrastructure

Identifiants uniques de la station (un regroupement de plusieurs bornes).
Usage final : Comptez le nombre de stations par commune (plutôt que le nombre de points de charge) pour mesurer le maillage territorial.
 'id_station_itinerance',
 'id_station_local',

Identifiants du point de charge précis.
'id_pdc_itinerance',
 'id_pdc_local',

Type de lieu (voirie, parking public, parking privé, etc.).
Usage final : Calculez la part des bornes en voirie (accessible 24/24) vs en parking.
 'implantation_station',

Nombre de points de charge sur la borne.
Usage final : À sommer par commune pour obtenir la capacité totale d'accueil.
 'nbre_pdc',


3. Puissance et Types de prises (Performance technique)
C'est le cœur de l'attractivité pour l'utilisateur.

Puissance délivrée (en kW).
Usage final : Calculez la puissance moyenne ou créez une variable "Nombre de bornes ultra-rapides" (> 50 kW).
 'puissance_nominale',

Indiquent les standards de recharge disponibles.
Usage final : Calculez le taux de compatibilité Combo CCS (standard européen pour la charge rapide). Une commune n'ayant que du Type EF (prises domestiques) est moins attractive.
 'prise_type_ef',
 'prise_type_2',
 'prise_type_combo_ccs',
 'prise_type_chademo',
 'prise_type_autre',
 'cable_t2_attache',


4. Services, Tarification et Accessibilité
Ces variables influencent directement le coût d'usage.

Indique si la recharge est gratuite.
Usage final : Taux de gratuité par commune. Un fort taux peut booster l'achat de VE locaux.
 'gratuit',

Modalités de paiement.
Usage final : Taux d'acceptation CB. Le paiement direct est un facteur clé de simplicité.
 'paiement_acte',
 'paiement_cb',
 'paiement_autre',
 'tarification',

Accessibilité de la borne.
Usage final : Créez une variable binaire sur la disponibilité 24h/24.
 'condition_acces',
 'reservation',
 'horaires',

Date de création de la fiche.
Usage final : Calculez l'ancienneté moyenne du réseau par commune.
 'created_at']

In [ ]:
# ----------------------------
# Statistiques générales
# ----------------------------
print("===== Statistiques =====")
print(df_filtre["puissance_nominale"].describe())

# ----------------------------
# Valeurs les plus fréquentes
# ----------------------------
print("===== Top puissances =====")
print(df_filtre["puissance_nominale"].value_counts().head(20))

# ----------------------------
# Histogramme global
# ----------------------------
plt.figure(figsize=(10,5))
df_filtre["puissance_nominale"].hist(bins=50)
plt.title("Distribution des puissances nominales")
plt.xlabel("Puissance (kW)")
plt.ylabel("Nombre de bornes")
plt.show()

# ----------------------------
# Zoom sur faibles puissances
# ----------------------------
plt.figure(figsize=(10,5))
df_filtre[df_filtre["puissance_nominale"] <= 100]["puissance_nominale"].hist(bins=40)
plt.title("Zoom sur puissances <= 100 kW")
plt.xlabel("Puissance (kW)")
plt.ylabel("Nombre")
plt.show()

# ----------------------------
# Boxplot
# ----------------------------
plt.figure(figsize=(10,3))
plt.boxplot(df_filtre["puissance_nominale"].dropna(), vert=False)
plt.title("Boxplot puissance nominale")
plt.show()

# ----------------------------
# Quantiles utiles
# ----------------------------
print("===== Quantiles =====")
print(df_filtre["puissance_nominale"].quantile([0.25,0.5,0.75,0.90,0.95,0.99]))

# ----------------------------
# Parts selon seuils métier
# ----------------------------
for seuil in [7, 22, 43, 50, 100, 150]:
    pct = (df_filtre["puissance_nominale"] >= seuil).mean() * 100
    print(f"% bornes >= {seuil} kW : {pct:.2f}%")

Une borne de recharge est considérée comme rapide lorsqu'elle est capable de proposer une puissance supérieure à 43 kW.